# Install Packages and Setup Variables



In [30]:
!pip install -q openai==1.93.0 cohere==5.15.0 tiktoken==0.8.0
!pip install PyPDF2
!pip install -q streamlit
!pip install -q pyngrok


# Import Packages and setup API keys

In [ ]:
import os
from pyngrok import ngrok #Hosting Streamlit
import subprocess
import time

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
ngrok.set_auth_token('YOUR NGROK_AUTH_TOKEN')


In [32]:
# False: Generate the embedding for the dataset. (Associated cost with using OpenAI endpoint)
# True: Load the dataset that already has the embedding vectors.
load_embedding = False

## Read File

In [33]:
# Split the input text into chunks of specified size.
def split_into_chunks(text, chunk_size=1024):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i : i + chunk_size])

    return chunks

In [34]:
import PyPDF2
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text() or ""
    return text

pdf_text = extract_text_from_pdf('/content/Management-of-type-2-diabetes.pdf')

# Clear existing chunks and populate with PDF content
chunks = []
chunks.extend(split_into_chunks(pdf_text))

# Convert the list of chunks to a Pandas DataFrame
df = pd.DataFrame(chunks, columns=["chunk"])
df.keys()

Index(['chunk'], dtype='object')

## Generate Embedding

In [35]:
from openai import OpenAI

client = OpenAI()


# Defining a function that converts a text to embedding vector using OpenAI's Ada model.
def get_embedding(text):
    try:
        # Remove newlines
        text = text.replace("\n", " ")
        res = client.embeddings.create(input=[text], model="text-embedding-3-small")

        return res.data[0].embedding

    except:
        return None

In [36]:
print("Generating embeddings for the PDF content...")
embeddings = []
for index, row in tqdm(df.iterrows()):
    embeddings.append(get_embedding(row["chunk"]))

embeddings_values = pd.Series(embeddings)
df.insert(loc=1, column="embedding", value=embeddings_values)

Generating embeddings for the PDF content...


658it [02:22,  4.60it/s]


## User Question

In [37]:
QUESTION = "What are the risk factors for Type 2 Diabetes and what preventative measures can be taken?"
QUESTION_emb = get_embedding(QUESTION)

len(QUESTION_emb)

1536

## Calculate Cosine Similarities

In [38]:
# The similarity between the questions and each part of the essay.
cosine_similarities = cosine_similarity([QUESTION_emb], df["embedding"].tolist())

print(cosine_similarities)

[[0.43196455 0.60459919 0.53452208 0.48386883 0.4878496  0.16073618
  0.32650957 0.351846   0.34490619 0.32564986 0.41305732 0.46639654
  0.37757808 0.32336673 0.34519087 0.3943632  0.44972758 0.41298058
  0.39035906 0.46317682 0.51872438 0.49296333 0.42227704 0.46833669
  0.40925089 0.39912753 0.22527285 0.25816665 0.40435497 0.34241063
  0.33665407 0.45292375 0.47813685 0.54661995 0.51771517 0.54531756
  0.52306489 0.37693899 0.53803105 0.49782768 0.52621545 0.58327647
  0.53244247 0.44489859 0.49497729 0.43596464 0.4717147  0.46510035
  0.52554782 0.37963665 0.47557966 0.44229932 0.41183703 0.45577051
  0.49276288 0.33040496 0.38137338 0.4815204  0.48051027 0.3909202
  0.33213325 0.39814232 0.43458706 0.49519765 0.50896324 0.44711541
  0.30575439 0.46726863 0.38560658 0.36015287 0.46646521 0.38125586
  0.45396878 0.47213662 0.43437873 0.39844759 0.41428067 0.38005123
  0.5112596  0.38561705 0.37262356 0.43314372 0.38474051 0.41454601
  0.45650847 0.46510934 0.38489751 0.41825234 0.4

# Pick N best Matched chunks

In [39]:
import numpy as np

number_of_chunks_to_retrieve = 7

# Sort the scores
highest_index = np.argmax(cosine_similarities)

# Pick the N highest scored chunks
indices = np.argsort(cosine_similarities[0])[::-1][:number_of_chunks_to_retrieve]
print(indices)

[222 138  96 117 134 141   1]


In [40]:
# Look at the highest scored retrieved pieces of text
for idx, item in enumerate(df.chunk[indices]):
    print(f"> Chunk {idx+1}")
    print(item)
    print("----")

> Chunk 1
 type 2
diabetes: Re view with meta-analysis of clinical
studies. J Am Coll Nutr 2003;22(5):331–39. doi:
10.1080/07315724.2003.10719316.
10. Bazzano L A, Ser dula M, Liu S. Pr evention of type
2 diabetes b y diet and lif estyle modification. J
Am Coll Nutr 2005;24(5):310–19. doi: 10.1080/
07315724.2005.10719479.
11. Knowler WC, Barr ett-Connor E, F owler SE, et al.
Reduction in the incidence of type 2 diabetes with
lifestyle inter vention or metformin. N Engl J Med
2002;346(6):393–403. doi: 10.1056/
NEJMoa012512.
12. Tuomileht o J, Lindstr öm J, Eriksson JG, et al.
Prevention of type 2 diabetes mellitus b y changes
in lif estyle among subjects with impair ed glucose
tolerance. N Engl J Med 2001;344(18):1343–50.
doi: 10.1056/NEJM200105033441801.
13. Knowler WC, F owler SE, Hamman RF , et al.
10-Y ear follow-up of diabetes incidence and
weight loss in the Diabetes Pr evention Pr ogram
Outcomes Study . Lancet
2009;374(9702):1677–86. doi: 10.1016/
S0140-6736(09)61457-4.
14. Wing 

## Test Cosine Similarity

Calculating the similarity of embedding representations can help us to find pieces of text that are close to each other. In the following sample you see how the Cosine Similarity metric can identify which sentence could be a possible answer for the given user question. Obviously, the unrelated answer will score lower.


In [41]:
BAD_SOURCE_emb = get_embedding("The history of space exploration is fascinating.")
GOOD_SOURCE_emb = get_embedding("Type 2 diabetes is often diagnosed through blood tests that measure blood glucose levels.")

In [42]:
from sklearn.metrics.pairwise import cosine_similarity

# to a completely unrelated text.
print("> Bad Response Score:", cosine_similarity([QUESTION_emb], [BAD_SOURCE_emb]))
print("> Good Response Score:", cosine_similarity([QUESTION_emb], [GOOD_SOURCE_emb]))

> Bad Response Score: [[-0.00637081]]
> Good Response Score: [[0.50881861]]


## Streamlit Interface

In [43]:
%%writefile app.py

import streamlit as st
import os
from openai import OpenAI
from google.colab import userdata

# Set the OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

client = OpenAI()

# Define the get_chat_completion function
def get_chat_completion(question, context):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are an assistant and expert in answering questions from a given context. If the answer is not in the context, state that you cannot answer the question based on the provided information."},
            {"role": "user", "content": f"Question: {question}\n\nContext: {context}\n\nAnswer:"}
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content

# Streamlit App Title
st.title("RAG Chatbot - Management of Type 2 Diabetes")

st.markdown("Ask a question about Management of Type 2 Diabetes in Australia")

# Input fields
question_input = st.text_input("Question", value="What are the risk factors for Type 2 Diabetes?")

# Load retrieved_context if available from environment variables (hidden from user)
context_from_env = os.getenv('RETRIEVED_CONTEXT', '')


if st.button("Get Answer"):
    if question_input:
        with st.spinner("Generating response..."):
            response = get_chat_completion(question_input, context_from_env)
            st.subheader("Answer:")
            st.write(response)
    else:
        st.warning("Please enter a question.")

Overwriting app.py


In [44]:
# Define the get_chat_completion function in the notebook's scope
def get_chat_completion(question, context):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are an assistant and expert in answering questions from a given context. If the answer is not in the context, state that you cannot answer the question based on the provided information."},
            {"role": "user", "content": f"Question: {question}\n\nContext: {context}\n\nAnswer:"}
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content

In [45]:
retrieved_context = "\n".join(df.chunk[indices])
response = get_chat_completion(QUESTION, retrieved_context)
print(response)

Risk factors for Type 2 Diabetes include demographic and social factors such as age, family history, and ethnicity, lifestyle factors like obesity, physical inactivity, and smoking, clinical history including high blood pressure, high triglycerides, low high-density lipoprotein cholesterol (HDL-C), gestational diabetes, heart disease, stroke, depression, polycystic ovary syndrome, acanthosis nigricans, and metabolic-associated fatty liver disease (MALFD), as well as medications like corticosteroids and antipsychotic medications.

Preventative measures that can be taken to reduce the risk of Type 2 Diabetes include lifestyle interventions such as increasing physical activity to at least 150 minutes per week, achieving and maintaining a 7% reduction in weight if overweight or obese, and making dietary changes. Additionally, cardiovascular fitness has been shown to decrease the risk of progression to Type 2 Diabetes. For individuals at high risk, lifestyle interventions and coaching focus

In [46]:

# Set the retrieved_context as an environment variable for the Streamlit app
os.environ["RETRIEVED_CONTEXT"] = retrieved_context

# Kill any running Streamlit process
!kill -9 $(lsof -t -i:8501)

# Start Streamlit in a separate process
process = subprocess.Popen(["streamlit", "run", "app.py"])

# Give Streamlit a moment to start
time.sleep(5)

# Open a ngrok tunnel to the Streamlit port
public_url = ngrok.connect(8501)

print(f"Your Streamlit app is running at: {public_url}")
print("To stop the Streamlit app and ngrok tunnel, run ngrok.kill() and then kill the Streamlit process (you might need to restart the Colab runtime if it doesn't stop cleanly).")

Your Streamlit app is running at: NgrokTunnel: "https://slobbery-raylene-dissyllabic.ngrok-free.dev" -> "http://localhost:8501"
To stop the Streamlit app and ngrok tunnel, run ngrok.kill() and then kill the Streamlit process (you might need to restart the Colab runtime if it doesn't stop cleanly).
